# Company Enrichment by Domain

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/company-enrichment-by-domain.ipynb)

**What:** Company enrichment from each company's own site and DNS only, verified by 73 passing tests. Domains in; name, logo, description, socials, careers/ATS, business emails and phones, email provider (MX) and tech stack out. No login, no data brokers. $0.005 per domain.

**Why:** run the [`company-enrichment-by-domain`](https://apify.com/transparent_meteorite/company-enrichment-by-domain) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

**Privacy:** Email, phone and social-handle columns are dropped before display; only company-level fields are shown.

## 1. Setup

In [1]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

Token loaded.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/company-enrichment-by-domain) for every option.

In [2]:
run_input = {'domains': ['basecamp.com', 'plausible.io'], 'maxPagesPerDomain': 3, 'includeTechStack': True}

## 3. Run the actor and read the dataset

In [3]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/company-enrichment-by-domain').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

[apify.company-enrichment-by-domain runId:okW9GehR2HDkYT6qE] -> Status: RUNNING, Message: 


[apify.company-enrichment-by-domain runId:okW9GehR2HDkYT6qE] -> 2026-10-13T22:17:00.780Z ACTOR: Pulling container image of build 2lBkEQWiD09hFYIp6 from registry.


[apify.company-enrichment-by-domain runId:okW9GehR2HDkYT6qE] -> 2026-10-13T22:17:01.691Z ACTOR: Creating container.


[apify.company-enrichment-by-domain runId:okW9GehR2HDkYT6qE] -> 2026-10-13T22:17:01.756Z ACTOR: Starting container.


[apify.company-enrichment-by-domain runId:okW9GehR2HDkYT6qE] -> 2026-10-13T22:17:01.757Z ACTOR: Running under "LIMITED_PERMISSIONS".


...

[apify.company-enrichment-by-domain runId:okW9GehR2HDkYT6qE] -> 2026-10-13T22:17:03.650Z [enrich] done: 2 ok, 0 not billable, 0 not attempted


[apify.company-enrichment-by-domain runId:okW9GehR2HDkYT6qE] -> 2026-10-13T22:17:03.719Z INFO  [Status message]: Done: 2 items


[apify.company-enrichment-by-domain runId:okW9GehR2HDkYT6qE] -> 2026-10-13T22:17:03.757Z Thanks for using Company Enrichment by Domain. Found a bug or need a field? Open an issue on the Issues tab (https://console.apify.com/actors/R87qiQPotD16ppa9B/issues) with the run URL and we will reply there.


[apify.company-enrichment-by-domain runId:okW9GehR2HDkYT6qE] -> Status: SUCCEEDED, Message: Done: 2 items


Run status: SUCCEEDED | items: 2


## 4. Tabulate

Email, phone and social-handle columns are dropped before display; only company-level fields are shown.

In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)
df['techCount'] = df['techStack'].apply(lambda t: t.get('count') if isinstance(t, dict) else None)

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["domain", "status", "companyName", "description", "language", "emailProvider", "spfPresent", "pagesFetched", "techCount"] if c in df.columns]
df[SHOW]

,domain,status,companyName,description,language,emailProvider,spfPresent,pagesFetched,techCount
0,plausible.io,ok,Plausible Analytics,Plausible is a lightweight and open-source Google Analytics altern...,en,other,True,3,5
1,basecamp.com,ok,Basecamp,"Trusted by millions, Basecamp puts everything you need to get work...",en,other,True,2,3


## 5. A quick analysis

In [5]:
print(df[['domain','techCount']])
print(df['emailProvider'].value_counts(dropna=False))

         domain  techCount
0  plausible.io          5
1  basecamp.com          3
emailProvider
other    2
Name: count, dtype: int64


## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [company-enrichment-by-domain on the Apify Store](https://apify.com/transparent_meteorite/company-enrichment-by-domain).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.